# Train the CNN guidance-map (Colab / GPU) — turnkey

Consumes `guidance_dataset.npz` built on the planner machine via
`ml_planner.dataset_gen.export_dataset` (or the bounded generator).

**Hard I/O contract (must match `ml_planner/guidance.py`):** input `channels`
shape `(1,4,256,256)` float32, output `cost_to_go` shape `(1,1,256,256)`
float32, opset >= 11, names exactly `channels` / `cost_to_go`.

Steps: upload the dataset, normalize labels, train a small U-Net with a
**masked MSE** (labeled cells only), export **onnx**, download `guidance.onnx`,
and drop it into `ml_planner/models/` on the planner machine — then
`plan_trajectory_focal(pre, secondary='guidance')` uses it automatically.

Runtime: set **Runtime -> Change runtime type -> GPU** (T4 is plenty).


In [ ]:
# 1. Load the dataset (upload guidance_dataset.npz when prompted, or mount Drive).
import numpy as np
try:
    from google.colab import files
    up = files.upload()                 # pick guidance_dataset.npz
    DATA_PATH = list(up.keys())[0]
except Exception:
    DATA_PATH = 'guidance_dataset.npz'
d = np.load(DATA_PATH)
channels = d['channels'].astype('float32')   # (N,4,256,256)
label    = d['label'].astype('float32')      # (N,256,256) cost-to-go, meters
mask     = d['mask'].astype('float32')       # (N,256,256) 1 where labeled
affine   = d['affine']                       # (N,4): x0, y0, scale, grid_res
N, C, H, W = channels.shape
assert (C, H, W) == (4, 256, 256), f'unexpected shape {channels.shape}'
print('samples', N, '| grid', H, W, '| labeled cells total', int(mask.sum()))


In [ ]:
# 2. Normalize cost-to-go by each sample's crop diagonal so targets are ~O(1).
# The planner uses the field only to RANK nodes per problem, so per-sample
# scaling is safe (ranking is scale-invariant) and greatly stabilizes training.
side = affine[:, 3] / affine[:, 2]                 # grid_res / scale = crop side (m)
diag = (np.sqrt(2.0) * side).astype('float32')     # (N,)
label_n = label / diag[:, None, None]
rng = np.random.default_rng(0)
idx = rng.permutation(N)
nval = max(1, N // 5)
val_idx, tr_idx = idx[:nval], idx[nval:]
print('train', len(tr_idx), 'val', len(val_idx))


In [ ]:
# 3. Small fully-convolutional U-Net (4 channels -> 1 cost-to-go field).
import torch, torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(co, co, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, cin=4, base=32):
        super().__init__()
        self.d1 = DoubleConv(cin, base);      self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(base, base*2);   self.p2 = nn.MaxPool2d(2)
        self.mid = DoubleConv(base*2, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.c2 = DoubleConv(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c1 = DoubleConv(base*2, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        x1 = self.d1(x); x2 = self.d2(self.p1(x1)); m = self.mid(self.p2(x2))
        y = self.c2(torch.cat([self.u2(m), x2], 1))
        y = self.c1(torch.cat([self.u1(y), x1], 1))
        return self.head(y)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNetSmall().to(dev)
print('device', dev)


In [ ]:
# 4. Train with masked MSE (only labeled cells contribute); keep best val model.
Xtr = torch.tensor(channels[tr_idx]); Ytr = torch.tensor(label_n[tr_idx]); Mtr = torch.tensor(mask[tr_idx])
Xva = torch.tensor(channels[val_idx]); Yva = torch.tensor(label_n[val_idx]); Mva = torch.tensor(mask[val_idx])
opt = torch.optim.Adam(model.parameters(), 1e-3)

def masked_mse(pred, y, m):
    m = m > 0
    return (((pred - y) ** 2) * m).sum() / m.sum().clamp(min=1)

best, best_state, bs = 1e9, None, 8
for epoch in range(60):
    model.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), bs):
        j = perm[i:i+bs]
        loss = masked_mse(model(Xtr[j].to(dev))[:, 0], Ytr[j].to(dev), Mtr[j].to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        vp = torch.cat([model(Xva[k:k+bs].to(dev))[:, 0].cpu() for k in range(0, len(Xva), bs)])
        vloss = float(masked_mse(vp, Yva, Mva))
    if vloss < best:
        best = vloss; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0:
        print(f'epoch {epoch:2d}  val masked-MSE {vloss:.5f}')
print('best val masked-MSE', best)
model.load_state_dict(best_state)


In [ ]:
# 5. Export to ONNX (contract: channels -> cost_to_go, 256x256) and download.
model.eval().cpu()
dummy = torch.zeros(1, 4, 256, 256)
torch.onnx.export(
    model, dummy, 'guidance.onnx',
    input_names=['channels'], output_names=['cost_to_go'],
    opset_version=13,
    dynamic_axes={'channels': {0: 'batch'}, 'cost_to_go': {0: 'batch'}})
print('exported guidance.onnx')
try:
    from google.colab import files
    files.download('guidance.onnx')
except Exception:
    pass
# Place guidance.onnx into ml_planner/models/ on the planner machine.
# secondary='guidance' then activates automatically (falls back to hand-crafted if absent).
